# GenIDS-CIC17: organization of NFStream flows

This notebook consolidates the daily CIC-IDS2017 flow files previously extracted from the original PCAP captures with NFStream. It standardizes the labels, removes duplicate rows, and exports the file used to build GenIDS-CIC17. Only the paths and, if necessary, the input filenames in the configuration cell should be changed.

## 1. Configuration

In [ ]:
from pathlib import Path

INPUT_DIR = Path("/path/to/cic17/nfstream_daily_csv")
OUTPUT_DIR = Path("/path/to/output")
OUTPUT_FILE = OUTPUT_DIR / "GenIDS-CIC17.csv"

DAILY_FILES = {
    "Monday": "01_nfstream_monday.csv",
    "Tuesday": "02_nfstream_tuesday.csv",
    "Wednesday": "03_nfstream_wednesday.csv",
    "Thursday": "04_nfstream_thursday.csv",
    "Friday": "05_nfstream_friday.csv",
}

REQUIRED_COLUMNS = {"id", "Timestamp", "binary", "multiclass"}
DDOS_LABELS = {
    "dos_hulk",
    "ddos",
    "dos_slowhttptest",
    "dos_goldeneye",
    "dos_slowloris",
}

## 2. Imports and helper functions

In [ ]:
import pandas as pd


def validate_input_files(input_dir, daily_files):
    missing = [input_dir / name for name in daily_files.values() if not (input_dir / name).is_file()]
    if missing:
        formatted = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(f"Required input files were not found:\n{formatted}")


def load_daily_flows(input_dir, daily_files, required_columns):
    frames = []
    summary = []
    for day, filename in daily_files.items():
        path = input_dir / filename
        frame = pd.read_csv(path, low_memory=False)
        missing_columns = required_columns.difference(frame.columns)
        if missing_columns:
            raise ValueError(f"{filename} is missing columns: {sorted(missing_columns)}")
        frames.append(frame)
        summary.append({"day": day, "file": filename, "flows": len(frame)})
    return pd.concat(frames, ignore_index=True), pd.DataFrame(summary)


def map_multiclass_label(label):
    normalized = str(label).strip().lower()
    if normalized == "benign":
        return "benign"
    if normalized in DDOS_LABELS:
        return "ddos"
    return "background"


def prepare_cic17_dataset(frame):
    prepared = frame.copy()
    timestamp = prepared["Timestamp"].astype(str).str.split(n=1, expand=True)
    if timestamp.shape[1] < 2:
        raise ValueError("Timestamp values must contain both date and time.")
    prepared["date"] = timestamp[0]
    prepared["hours"] = timestamp[1]
    prepared["multiclass"] = prepared["multiclass"].map(map_multiclass_label)
    prepared = prepared.drop(columns=["id", "Timestamp"])
    prepared = prepared.drop_duplicates().reset_index(drop=True)
    label_columns = ["binary", "multiclass"]
    feature_columns = [column for column in prepared.columns if column not in label_columns]
    return prepared[feature_columns + label_columns]

## 3. Load and consolidate daily flow files

In [ ]:
validate_input_files(INPUT_DIR, DAILY_FILES)
flows, daily_summary = load_daily_flows(INPUT_DIR, DAILY_FILES, REQUIRED_COLUMNS)

display(daily_summary)
print(f"Total flows loaded: {len(flows):,}")
print(f"Total columns: {flows.shape[1]}")

## 4. Standardize labels and remove duplicates

In [ ]:
rows_before = len(flows)
genids_cic17 = prepare_cic17_dataset(flows)
rows_after = len(genids_cic17)

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal:  {rows_after:,}")
print(f"Duplicate rows removed:        {rows_before - rows_after:,}")

## 5. Dataset summary

In [ ]:
binary_summary = pd.DataFrame({
    "count": genids_cic17["binary"].value_counts(),
    "percentage": genids_cic17["binary"].value_counts(normalize=True).mul(100).round(2),
})
multiclass_summary = pd.DataFrame({
    "count": genids_cic17["multiclass"].value_counts(),
    "percentage": genids_cic17["multiclass"].value_counts(normalize=True).mul(100).round(2),
})

display(binary_summary)
display(multiclass_summary)
display(genids_cic17.head())

## 6. Export

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
genids_cic17.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {genids_cic17.shape}")